# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/vascular`. Старые метрики не смешиваются с актуальными.


# Статистический анализ госпитальных исходов

Анализ является описательным и исследовательским. Прогнозные и регрессионные модели в ноутбуке не строятся. Основной и импутированный наборы анализируются раздельно.

Уровень глюкозы анализируется как непрерывный показатель и как бинарный прокси-признак повышения при значении ≥5,6 ммоль/л. Статус измерения натощак в источнике не указан.


## 1. Настройки и загрузка данных

In [ ]:
from pathlib import Path
import os
import re

RESULT_DIR = Path("hospital_outcomes_statistical_results")
TABLE_DIR = RESULT_DIR / "tables"
FIGURE_DIR = RESULT_DIR / "figures"
MPL_CACHE_DIR = RESULT_DIR / ".matplotlib"

for directory in [RESULT_DIR, TABLE_DIR, FIGURE_DIR, MPL_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for output_file in [
    *TABLE_DIR.glob("*.csv"),
    *FIGURE_DIR.glob("*.png"),
]:
    output_file.unlink()

os.environ["MPLCONFIGDIR"] = str(MPL_CACHE_DIR.resolve())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from scipy.stats import mannwhitneyu, fisher_exact
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")


In [ ]:
DATA_DIR = Path("data")

dataset_specs = {
    "prepared": {
        "label": "Общая выборка",
        "path": DATA_DIR / "hospital_outcomes_prepared.csv",
    },
    "peripheral": {
        "label": "Периферические артерии",
        "path": DATA_DIR / "hospital_outcomes_peripheral.csv",
    },
    "absh_aortic_aneurysm": {
        "label": "АБШ / аневризма аорты",
        "path": DATA_DIR / "hospital_outcomes_absh_aortic_aneurysm.csv",
    },
    "carotid": {
        "label": "Сонные артерии",
        "path": DATA_DIR / "hospital_outcomes_carotid.csv",
    },
    "imputed_exploratory": {
        "label": "Импутированный исследовательский набор",
        "path": DATA_DIR / "hospital_outcomes_imputed_exploratory.csv",
    },
}

datasets = {
    key: pd.read_csv(specification["path"])
    for key, specification in dataset_specs.items()
}

for key, frame in datasets.items():
    frame["hospital_death"] = pd.to_numeric(
        frame["hospital_death"],
        errors="raise"
    ).astype("Int64")

## 2. Переменные и статистические функции

In [ ]:
continuous_features = [
    "age_years",
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

binary_features = [
    "female",
    "prior_vascular_surgery",
    "mi_history",
    "stroke_history",
    "ckd_g3plus_proxy",
    "glucose_elevated_proxy",
]

imputation_targets = [
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

variable_labels = {
    "age_years": "Возраст, лет",
    "ef_percent": "Фракция выброса, %",
    "creatinine_value": "Креатинин",
    "total_cholesterol": "Общий холестерин",
    "ldl_mmol_l": "ЛПНП, ммоль/л",
    "glucose_value": "Глюкоза, ммоль/л",
    "crp_value": "С-реактивный белок",
    "egfr_value": "СКФ",
    "bmi": "Индекс массы тела",
    "female": "Женский пол",
    "prior_vascular_surgery": "Операция на сосудах в анамнезе",
    "mi_history": "Инфаркт миокарда в анамнезе",
    "stroke_history": "ОНМК в анамнезе",
    "ckd_g3plus_proxy": "ХБП G3 и выше, прокси",
    "glucose_elevated_proxy": "Повышенная глюкоза ≥5,6 ммоль/л, прокси",
    "ef_category": "Категория ФВ",
}


In [ ]:
def wilson_interval(events, total, confidence_z=1.959963984540054):
    if total == 0:
        return np.nan, np.nan
    probability = events / total
    denominator = 1 + confidence_z**2 / total
    center = (
        probability
        + confidence_z**2 / (2 * total)
    ) / denominator
    margin = (
        confidence_z
        * np.sqrt(
            probability * (1 - probability) / total
            + confidence_z**2 / (4 * total**2)
        )
        / denominator
    )
    return center - margin, center + margin


def quantile_summary(series):
    observed = series.dropna()
    if observed.empty:
        return {
            "n": 0,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
        }
    return {
        "n": int(observed.size),
        "median": float(observed.median()),
        "q25": float(observed.quantile(0.25)),
        "q75": float(observed.quantile(0.75)),
    }


def odds_ratio_interval(a, b, c, d):
    values = np.array([a, b, c, d], dtype=float)
    if (values == 0).any():
        values = values + 0.5
    a_value, b_value, c_value, d_value = values
    odds_ratio = (a_value * d_value) / (b_value * c_value)
    standard_error = np.sqrt(
        1 / a_value
        + 1 / b_value
        + 1 / c_value
        + 1 / d_value
    )
    lower = np.exp(np.log(odds_ratio) - 1.959963984540054 * standard_error)
    upper = np.exp(np.log(odds_ratio) + 1.959963984540054 * standard_error)
    return odds_ratio, lower, upper


def add_fdr(table, p_column="p_value"):
    result = table.copy()
    result["p_fdr"] = np.nan
    valid = result[p_column].notna()
    if valid.any():
        result.loc[valid, "p_fdr"] = multipletests(
            result.loc[valid, p_column],
            method="fdr_bh"
        )[1]
    return result


In [ ]:
def continuous_outcome_analysis(frame):
    rows = []

    for variable in continuous_features:
        alive = frame.loc[
            frame["hospital_death"] == 0,
            variable
        ].dropna()
        deceased = frame.loc[
            frame["hospital_death"] == 1,
            variable
        ].dropna()

        alive_summary = quantile_summary(alive)
        deceased_summary = quantile_summary(deceased)

        if alive.size > 0 and deceased.size > 0:
            test = mannwhitneyu(
                deceased,
                alive,
                alternative="two-sided"
            )
            rank_biserial = (
                2 * test.statistic / (deceased.size * alive.size)
                - 1
            )
            p_value = float(test.pvalue)
        else:
            rank_biserial = np.nan
            p_value = np.nan

        rows.append(
            {
                "variable": variable,
                "label_ru": variable_labels[variable],
                "alive_n": alive_summary["n"],
                "alive_median": alive_summary["median"],
                "alive_q25": alive_summary["q25"],
                "alive_q75": alive_summary["q75"],
                "deceased_n": deceased_summary["n"],
                "deceased_median": deceased_summary["median"],
                "deceased_q25": deceased_summary["q25"],
                "deceased_q75": deceased_summary["q75"],
                "median_difference": (
                    deceased_summary["median"]
                    - alive_summary["median"]
                ),
                "rank_biserial": rank_biserial,
                "p_value": p_value,
            }
        )

    return add_fdr(pd.DataFrame(rows))


In [ ]:
def binary_outcome_analysis(frame, features=None):
    selected_features = features or binary_features
    rows = []

    for variable in selected_features:
        observed = frame[[variable, "hospital_death"]].dropna()

        a = int(
            (
                (observed[variable] == 1)
                & (observed["hospital_death"] == 1)
            ).sum()
        )
        b = int(
            (
                (observed[variable] == 1)
                & (observed["hospital_death"] == 0)
            ).sum()
        )
        c = int(
            (
                (observed[variable] == 0)
                & (observed["hospital_death"] == 1)
            ).sum()
        )
        d = int(
            (
                (observed[variable] == 0)
                & (observed["hospital_death"] == 0)
            ).sum()
        )

        exposed_total = a + b
        unexposed_total = c + d
        both_outcomes = (a + c) > 0 and (b + d) > 0
        both_exposures = exposed_total > 0 and unexposed_total > 0

        if both_outcomes and both_exposures:
            p_value = float(
                fisher_exact(
                    [[a, b], [c, d]],
                    alternative="two-sided"
                ).pvalue
            )
            odds_ratio, odds_ratio_lower, odds_ratio_upper = (
                odds_ratio_interval(a, b, c, d)
            )
        else:
            p_value = np.nan
            odds_ratio = np.nan
            odds_ratio_lower = np.nan
            odds_ratio_upper = np.nan

        rows.append(
            {
                "variable": variable,
                "label_ru": variable_labels.get(variable, variable),
                "available_n": int(observed.shape[0]),
                "exposed_n": exposed_total,
                "exposed_deaths": a,
                "exposed_mortality_pct": (
                    a / exposed_total * 100
                    if exposed_total > 0
                    else np.nan
                ),
                "unexposed_n": unexposed_total,
                "unexposed_deaths": c,
                "unexposed_mortality_pct": (
                    c / unexposed_total * 100
                    if unexposed_total > 0
                    else np.nan
                ),
                "odds_ratio": odds_ratio,
                "odds_ratio_ci_lower": odds_ratio_lower,
                "odds_ratio_ci_upper": odds_ratio_upper,
                "p_value": p_value,
            }
        )

    return add_fdr(pd.DataFrame(rows))


In [ ]:
def missingness_outcome_analysis(frame):
    rows = []

    for variable in continuous_features:
        missing = frame[variable].isna()

        if not missing.any():
            continue

        a = int((missing & (frame["hospital_death"] == 1)).sum())
        b = int((missing & (frame["hospital_death"] == 0)).sum())
        c = int((~missing & (frame["hospital_death"] == 1)).sum())
        d = int((~missing & (frame["hospital_death"] == 0)).sum())

        if (a + c) > 0 and (b + d) > 0 and (a + b) > 0 and (c + d) > 0:
            p_value = float(
                fisher_exact(
                    [[a, b], [c, d]],
                    alternative="two-sided"
                ).pvalue
            )
            odds_ratio, odds_ratio_lower, odds_ratio_upper = (
                odds_ratio_interval(a, b, c, d)
            )
        else:
            p_value = np.nan
            odds_ratio = np.nan
            odds_ratio_lower = np.nan
            odds_ratio_upper = np.nan

        rows.append(
            {
                "variable": variable,
                "label_ru": variable_labels[variable],
                "missing_n": a + b,
                "missing_deaths": a,
                "missing_mortality_pct": (
                    a / (a + b) * 100
                    if (a + b) > 0
                    else np.nan
                ),
                "observed_n": c + d,
                "observed_deaths": c,
                "observed_mortality_pct": (
                    c / (c + d) * 100
                    if (c + d) > 0
                    else np.nan
                ),
                "odds_ratio_missing": odds_ratio,
                "odds_ratio_ci_lower": odds_ratio_lower,
                "odds_ratio_ci_upper": odds_ratio_upper,
                "p_value": p_value,
            }
        )

    result = pd.DataFrame(rows)
    return add_fdr(result) if not result.empty else result


## 3. Общая структура выборок и госпитальная смертность

In [ ]:
dataset_overview_rows = []

for key, frame in datasets.items():
    patients = int(frame.shape[0])
    deaths = int(frame["hospital_death"].sum())
    lower, upper = wilson_interval(deaths, patients)

    dataset_overview_rows.append(
        {
            "dataset": key,
            "label_ru": dataset_specs[key]["label"],
            "patients": patients,
            "deaths": deaths,
            "mortality_pct": deaths / patients * 100,
            "mortality_ci95_lower_pct": lower * 100,
            "mortality_ci95_upper_pct": upper * 100,
            "columns": int(frame.shape[1]),
            "missing_cells": int(frame.isna().sum().sum()),
        }
    )

dataset_overview = pd.DataFrame(dataset_overview_rows)
dataset_overview.to_csv(
    TABLE_DIR / "dataset_overview.csv",
    index=False,
    encoding="utf-8-sig"
)

display(dataset_overview)


In [ ]:
primary_keys = [
    "prepared",
    "peripheral",
    "absh_aortic_aneurysm",
    "carotid",
]

mortality_plot_data = dataset_overview.loc[
    dataset_overview["dataset"].isin(primary_keys)
].copy()

mortality_plot_data["lower_error"] = (
    mortality_plot_data["mortality_pct"]
    - mortality_plot_data["mortality_ci95_lower_pct"]
)
mortality_plot_data["upper_error"] = (
    mortality_plot_data["mortality_ci95_upper_pct"]
    - mortality_plot_data["mortality_pct"]
)

figure, axis = plt.subplots(figsize=(10, 5))
axis.bar(
    mortality_plot_data["label_ru"],
    mortality_plot_data["mortality_pct"],
    color=["#355C7D", "#6C9A8B", "#C06C84", "#7B8FA1"]
)
axis.errorbar(
    mortality_plot_data["label_ru"],
    mortality_plot_data["mortality_pct"],
    yerr=[
        mortality_plot_data["lower_error"],
        mortality_plot_data["upper_error"],
    ],
    fmt="none",
    color="black",
    capsize=5
)
axis.set_ylabel("Госпитальная смертность, %")
axis.set_xlabel("")
axis.set_title("Госпитальная смертность и 95% доверительные интервалы")
axis.tick_params(axis="x", rotation=20)
figure.tight_layout()
figure.savefig(
    FIGURE_DIR / "mortality_by_dataset.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()
plt.close(figure)

### 3.1. Попарное сравнение смертности между сосудистыми группами

In [ ]:
vascular_group_pairs = [
    ("peripheral", "absh_aortic_aneurysm"),
    ("peripheral", "carotid"),
    ("absh_aortic_aneurysm", "carotid"),
]

group_comparison_rows = []

for first_group, second_group in vascular_group_pairs:
    first_frame = datasets[first_group]
    second_frame = datasets[second_group]

    a = int(first_frame["hospital_death"].sum())
    b = int(first_frame.shape[0] - a)
    c = int(second_frame["hospital_death"].sum())
    d = int(second_frame.shape[0] - c)

    test = fisher_exact(
        [[a, b], [c, d]],
        alternative="two-sided"
    )
    odds_ratio, odds_ratio_lower, odds_ratio_upper = (
        odds_ratio_interval(a, b, c, d)
    )

    group_comparison_rows.append(
        {
            "first_group": first_group,
            "first_group_label_ru": dataset_specs[first_group]["label"],
            "first_patients": a + b,
            "first_deaths": a,
            "first_mortality_pct": a / (a + b) * 100,
            "second_group": second_group,
            "second_group_label_ru": dataset_specs[second_group]["label"],
            "second_patients": c + d,
            "second_deaths": c,
            "second_mortality_pct": c / (c + d) * 100,
            "odds_ratio_first_vs_second": odds_ratio,
            "odds_ratio_ci_lower": odds_ratio_lower,
            "odds_ratio_ci_upper": odds_ratio_upper,
            "p_value": float(test.pvalue),
        }
    )

vascular_group_comparison = add_fdr(
    pd.DataFrame(group_comparison_rows)
)

vascular_group_comparison.to_csv(
    TABLE_DIR / "vascular_group_mortality_pairwise.csv",
    index=False,
    encoding="utf-8-sig"
)

display(vascular_group_comparison)


## 4. Статистическое сравнение выживших и умерших

In [ ]:
continuous_results = {}
binary_results = {}

for key, frame in datasets.items():
    display(Markdown(f"### {dataset_specs[key]['label']}"))

    continuous_table = continuous_outcome_analysis(frame)
    binary_table = binary_outcome_analysis(frame)

    continuous_results[key] = continuous_table
    binary_results[key] = binary_table

    continuous_table.to_csv(
        TABLE_DIR / f"{key}_continuous_outcome.csv",
        index=False,
        encoding="utf-8-sig"
    )
    binary_table.to_csv(
        TABLE_DIR / f"{key}_binary_outcome.csv",
        index=False,
        encoding="utf-8-sig"
    )

    display(continuous_table)
    display(binary_table)


In [ ]:
overall_continuous_ranked = (
    continuous_results["prepared"]
    .sort_values(["p_fdr", "p_value"], na_position="last")
    .reset_index(drop=True)
)

overall_binary_ranked = (
    binary_results["prepared"]
    .sort_values(["p_fdr", "p_value"], na_position="last")
    .reset_index(drop=True)
)

display(Markdown("### Общая выборка: ранжирование результатов"))
display(overall_continuous_ranked)
display(overall_binary_ranked)


In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Конфигурация 5 графиков в нужном порядке (датасет + переменная)
# plot_specs = [
#     {'group': 'peripheral',           'var': 'age_years'},        # Периферические: возраст
#     {'group': 'peripheral',           'var': 'creatinine_value'}, # Периферические: креатинин
#     {'group': 'peripheral',           'var': 'crp_value'},        # Периферические: СРБ
#     {'group': 'absh_aortic_aneurysm', 'var': 'creatinine_value'}, # АБШ / аорта: креатинин
#     {'group': 'absh_aortic_aneurysm', 'var': 'glucose_value'}     # АБШ / аорта: глюкоза
# ]

# # Цвета
# palette = {
#     0: '#466c8b', 1: '#be7488', 
#     '0': '#466c8b', '1': '#be7488'
# }

# # Создаем сетку 2 на 3
# fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))
# axes = axes.flatten()

# # Рисуем графики в цикле
# for i, spec in enumerate(plot_specs):
#     ax = axes[i]
#     df_plot = datasets[spec['group']] # Берем нужную таблицу из вашего словаря
#     var = spec['var']
    
#     if var not in df_plot.columns:
#         continue
        
#     sns.boxplot(
#         data=df_plot,
#         x='hospital_death',
#         y=var,
#         hue='hospital_death',
#         palette=palette,
#         showfliers=False, # Убираем точки-выбросы
#         width=0.6,
#         linewidth=1.2,
#         legend=False,
#         ax=ax
#     )
    
#     # УБИРАЕМ ВСЕ СЛОВА
#     ax.set_title('')       # Убираем заголовок ("Периферические: возраст" и т.д.)
#     ax.set_xlabel('')      # Убираем подпись оси X
#     ax.set_ylabel('')      # Убираем подпись оси Y ("лет", "ммоль/л")
#     ax.set_xticks([])      # Полностью убираем текстовые подписи ("Живы", "Умерли")
    
#     # Визуальный тюнинг
#     sns.despine(ax=ax)
#     ax.yaxis.grid(True, color='lightgray', linestyle='-', alpha=0.7)
#     ax.set_axisbelow(True)
    
# # Удаляем 6-й пустой слот
# fig.delaxes(axes[5])

# plt.tight_layout()
# plt.show()



import matplotlib.pyplot as plt
import seaborn as sns

# Конфигурация 5 графиков в нужном порядке
plot_specs = [
    {'group': 'peripheral',           'var': 'age_years'},        
    {'group': 'peripheral',           'var': 'creatinine_value'}, 
    {'group': 'peripheral',           'var': 'crp_value'},        
    {'group': 'absh_aortic_aneurysm', 'var': 'creatinine_value'}, 
    {'group': 'absh_aortic_aneurysm', 'var': 'glucose_value'}     
]

# Цвета
palette = {
    0: '#466c8b', 1: '#be7488', 
    '0': '#466c8b', '1': '#be7488'
}

# Создаем сетку 2 на 3
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))
axes = axes.flatten()

# Рисуем графики в цикле
for i, spec in enumerate(plot_specs):
    ax = axes[i]
    df_plot = datasets[spec['group']] # Берем нужную таблицу из вашего словаря
    var = spec['var']
    
    if var not in df_plot.columns:
        continue
        
    sns.boxplot(
        data=df_plot,
        x='hospital_death',
        y=var,
        hue='hospital_death',
        order=[0, 1], # Явно фиксируем порядок: 0 (Живы) слева, 1 (Умерли) справа
        palette=palette,
        showfliers=False, 
        width=0.6,
        linewidth=1.2,
        legend=False,
        ax=ax
    )
    
    # 1. Подсчет реального числа пациентов (n) без учета пропусков
    n_alive = df_plot.loc[df_plot['hospital_death'] == 0, var].count()
    n_dead = df_plot.loc[df_plot['hospital_death'] == 1, var].count()
    
    # 2. Убираем слова с осей и заголовков
    ax.set_title('')       
    ax.set_xlabel('')      
    ax.set_ylabel('')      
    
    # 3. ДОБАВЛЯЕМ ПОДПИСИ "n=..." КАК НА ФОТО
    ax.set_xticks([0, 1])
    # Если хотите убрать слова "Живы/Умерли" и оставить ТОЛЬКО "n=...", 
    # замените строку ниже на: ax.set_xticklabels([f'n={n_alive}', f'n={n_dead}'])
    ax.set_xticklabels([f'n = {n_alive}', f'n = {n_dead}'])
    
    # Визуальный тюнинг
    sns.despine(ax=ax)
    ax.yaxis.grid(True, color='lightgray', linestyle='-', alpha=0.7)
    ax.set_axisbelow(True)

# Удаляем 6-й пустой слот
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('сohort_sample_boxplots.png', dpi=300, bbox_inches='tight')
fig.savefig('сohort_sample_boxplots.eps', format='eps', bbox_inches='tight')

## 5. Категории фракции выброса

In [ ]:
ef_category_tables = {}

for key, frame in datasets.items():
    category_table = (
        frame
        .dropna(subset=["ef_category"])
        .groupby(
            ["hospital_death", "ef_category"],
            observed=True
        )
        .size()
        .rename("n")
        .reset_index()
    )

    if not category_table.empty:
        category_table["percent_within_outcome"] = (
            category_table
            .groupby("hospital_death")["n"]
            .transform(lambda series: series / series.sum() * 100)
        )

    ef_category_tables[key] = category_table
    category_table.to_csv(
        TABLE_DIR / f"{key}_ef_categories.csv",
        index=False,
        encoding="utf-8-sig"
    )

    display(Markdown(f"### {dataset_specs[key]['label']}"))
    display(category_table)


## 6. Половое сравнение

In [ ]:
sex_summary_rows = []

for key in primary_keys:
    frame = datasets[key]

    for female_value, sex_label in [(0, "Мужчины"), (1, "Женщины")]:
        subset = frame.loc[frame["female"] == female_value]
        patients = int(subset.shape[0])
        deaths = int(subset["hospital_death"].sum())
        lower, upper = wilson_interval(deaths, patients)

        sex_summary_rows.append(
            {
                "dataset": key,
                "dataset_label_ru": dataset_specs[key]["label"],
                "sex": sex_label,
                "patients": patients,
                "deaths": deaths,
                "mortality_pct": (
                    deaths / patients * 100
                    if patients > 0
                    else np.nan
                ),
                "mortality_ci95_lower_pct": lower * 100,
                "mortality_ci95_upper_pct": upper * 100,
            }
        )

sex_summary = pd.DataFrame(sex_summary_rows)
sex_summary.to_csv(
    TABLE_DIR / "sex_mortality_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(sex_summary)


In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 9))

for axis, key in zip(axes.flat, primary_keys):
    plot_frame = sex_summary.loc[sex_summary["dataset"] == key]
    sns.barplot(
        data=plot_frame,
        x="sex",
        y="mortality_pct",
        hue="sex",
        palette={"Мужчины": "#355C7D", "Женщины": "#C06C84"},
        legend=False,
        ax=axis
    )
    axis.set_title(dataset_specs[key]["label"])
    axis.set_xlabel("")
    axis.set_ylabel("Смертность, %")

figure.suptitle("Госпитальная смертность по полу", y=1.02)
figure.tight_layout()
figure.savefig(
    FIGURE_DIR / "mortality_by_sex_and_dataset.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()
plt.close(figure)


## 7. Графики для каждого датасета

In [ ]:
def save_figure(figure, filename):
    figure.tight_layout()
    figure.savefig(
        FIGURE_DIR / filename,
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(figure)


def continuous_axes():
    columns = 3
    rows = int(np.ceil(len(continuous_features) / columns))
    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(18, 4.5 * rows)
    )
    return figure, np.asarray(axes).reshape(-1)


def hide_unused_axes(axes):
    for axis in axes[len(continuous_features):]:
        axis.set_visible(False)


def plot_continuous_distributions(frame, label, key):
    figure, axes = continuous_axes()
    has_two_outcomes = frame["hospital_death"].nunique() > 1

    for axis, variable in zip(axes, continuous_features):
        if has_two_outcomes:
            sns.histplot(
                data=frame,
                x=variable,
                hue="hospital_death",
                hue_order=[0, 1],
                palette={0: "#355C7D", 1: "#C06C84"},
                element="step",
                stat="density",
                common_norm=False,
                ax=axis
            )
        else:
            sns.histplot(
                data=frame,
                x=variable,
                color="#355C7D",
                ax=axis
            )
        axis.set_title(variable_labels[variable])
        axis.set_xlabel("")
        axis.set_ylabel("Плотность" if has_two_outcomes else "Частота")

    hide_unused_axes(axes)
    figure.suptitle(f"Распределения числовых показателей: {label}", y=1.02)
    save_figure(figure, f"{key}_continuous_distributions.png")


def plot_continuous_boxplots(frame, label, key):
    figure, axes = continuous_axes()
    has_two_outcomes = frame["hospital_death"].nunique() > 1

    for axis, variable in zip(axes, continuous_features):
        if has_two_outcomes:
            sns.boxplot(
                data=frame,
                x="hospital_death",
                y=variable,
                order=[0, 1],
                hue="hospital_death",
                palette={0: "#355C7D", 1: "#C06C84"},
                legend=False,
                ax=axis
            )
            axis.set_xticks([0, 1], ["Жив", "Умер"])
            axis.set_xlabel("")
        else:
            sns.boxplot(
                data=frame,
                y=variable,
                color="#355C7D",
                ax=axis
            )
            axis.set_xlabel("Только выжившие")
        axis.set_title(variable_labels[variable])
        axis.set_ylabel("")

    hide_unused_axes(axes)
    figure.suptitle(f"Показатели по госпитальному исходу: {label}", y=1.02)
    save_figure(figure, f"{key}_continuous_boxplots.png")

def plot_binary_prevalence(frame, label, key):
    rows = []

    for variable in binary_features:
        observed = frame[variable].dropna()
        rows.append(
            {
                "variable": variable,
                "label_ru": variable_labels[variable],
                "prevalence_pct": (
                    observed.eq(1).mean() * 100
                    if not observed.empty
                    else np.nan
                ),
            }
        )

    plot_frame = (
        pd.DataFrame(rows)
        .dropna(subset=["prevalence_pct"])
        .sort_values("prevalence_pct")
    )

    figure, axis = plt.subplots(figsize=(12, 5.5))
    sns.barplot(
        data=plot_frame,
        x="prevalence_pct",
        y="label_ru",
        hue="label_ru",
        palette="crest",
        legend=False,
        ax=axis
    )
    axis.set_xlabel("Распространённость среди доступных наблюдений, %")
    axis.set_ylabel("")
    axis.set_title(f"Бинарные признаки: {label}")
    save_figure(figure, f"{key}_binary_prevalence.png")


In [ ]:
for key, frame in datasets.items():
    label = dataset_specs[key]["label"]
    display(Markdown(f"### {label}"))
    plot_continuous_distributions(frame, label, key)
    plot_continuous_boxplots(frame, label, key)
    plot_binary_prevalence(frame, label, key)


## 8. Анализ импутированного датасета

In [ ]:
main_frame = datasets["prepared"]
imputed_frame = datasets["imputed_exploratory"]

imputation_comparison_rows = []

for variable in imputation_targets:
    indicator = f"{variable}_was_missing"
    observed_values = main_frame[variable].dropna()
    filled_values = imputed_frame.loc[
        imputed_frame[indicator] == 1,
        variable
    ]
    all_imputed_values = imputed_frame[variable]

    imputation_comparison_rows.append(
        {
            "variable": variable,
            "label_ru": variable_labels[variable],
            "observed_n": int(observed_values.size),
            "filled_n": int(filled_values.size),
            "observed_median": observed_values.median(),
            "observed_q25": observed_values.quantile(0.25),
            "observed_q75": observed_values.quantile(0.75),
            "filled_median": filled_values.median(),
            "filled_q25": filled_values.quantile(0.25),
            "filled_q75": filled_values.quantile(0.75),
            "after_median": all_imputed_values.median(),
            "after_q25": all_imputed_values.quantile(0.25),
            "after_q75": all_imputed_values.quantile(0.75),
        }
    )

imputation_comparison = pd.DataFrame(imputation_comparison_rows)
imputation_comparison.to_csv(
    TABLE_DIR / "imputation_distribution_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

display(imputation_comparison)


## 10. Технический контроль пропусков

Раздел вынесен в конец ноутбука и не относится к основным отчётным графикам. Здесь представлены пропуски числовых, бинарных и категориальных признаков, а также их исследовательская связь с госпитальным исходом.

In [ ]:
technical_variable_types = {
    **{variable: "Числовой" for variable in continuous_features},
    **{variable: "Бинарный" for variable in binary_features},
    "ef_category": "Категориальный",
}

technical_missingness_rows = []

for key, frame in datasets.items():
    for variable, variable_type in technical_variable_types.items():
        technical_missingness_rows.append(
            {
                "dataset": key,
                "dataset_label_ru": dataset_specs[key]["label"],
                "variable": variable,
                "label_ru": variable_labels.get(variable, variable),
                "variable_type": variable_type,
                "available_n": int(frame[variable].notna().sum()),
                "missing_n": int(frame[variable].isna().sum()),
                "missing_pct": round(frame[variable].isna().mean() * 100, 1),
            }
        )

technical_missingness = (
    pd.DataFrame(technical_missingness_rows)
    .sort_values(
        ["dataset", "variable_type", "missing_pct"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

technical_missingness.to_csv(
    TABLE_DIR / "technical_missingness_all_datasets.csv",
    index=False,
    encoding="utf-8-sig"
)

for key in datasets:
    display(Markdown(f"### {dataset_specs[key]['label']}"))
    display(
        technical_missingness.loc[
            technical_missingness["dataset"] == key
        ].reset_index(drop=True)
    )


In [ ]:
missingness_results = {}

for key, frame in datasets.items():
    missingness_table = missingness_outcome_analysis(frame)
    missingness_results[key] = missingness_table

    missingness_table.to_csv(
        TABLE_DIR / f"{key}_missingness_outcome.csv",
        index=False,
        encoding="utf-8-sig"
    )

    display(Markdown(f"### Пропуски и исход: {dataset_specs[key]['label']}"))
    display(missingness_table)

overall_missingness_ranked = (
    missingness_results["prepared"]
    .sort_values(["p_fdr", "p_value"], na_position="last")
    .reset_index(drop=True)
)

display(Markdown("### Общая выборка: ранжирование пропусков по связи с исходом"))
display(overall_missingness_ranked)


In [ ]:
result_inventory = pd.DataFrame(
    {
        "tables": sorted(path.name for path in TABLE_DIR.glob("*.csv")),
    }
)

print("Таблиц сохранено:", len(list(TABLE_DIR.glob("*.csv"))))
print("Графиков сохранено:", len(list(FIGURE_DIR.glob("*.png"))))
print("Каталог результатов:", RESULT_DIR.resolve())
